# Desafio AE — Etapa 1: Ingestão, Validação e EDA (AdventureWorks)

### Perguntas de Negócio a Serem Respondidas

a — Qual o número de pedidos, a quantidade comprada e o valor total da transação por produto, tipo de cartão, motivo da venda, data da venda, cliente, status, cidade, estado e país?

b — Quais produtos têm o maior valor médio por pedido, por mês, ano, cidade, estado e país? (valor médio por pedido = receita bruta − descontos de produto / número de pedidos no período analisado)

c — Quem são os 10 principais clientes por valor total de transação, filtrando por produto, tipo de cartão, motivo da venda, data da venda, status, cidade, estado e país?

d — Quais são as 5 principais cidades por valor total de transação, filtrando por produto, tipo de cartão, motivo da venda, data da venda, cliente, status, cidade, estado e país?

e — Qual o número de pedidos, a quantidade comprada e o valor total da transação por mês e ano? (dica: gráfico de série temporal)

f — Qual produto tem o maior número de unidades compradas pelo motivo de venda "Promotion"?

### Ingestão AdventureWorks — CSV para Delta

Pipeline de ingestão em arquitetura medallion (Bronze/Silver).

- **Bronze**: cópia fiel do dado bruto. Leitura textual integral, sem inferência
  de schema, preservando os valores exatamente como na origem.
- **Silver**: dado tipado e validado. Conversão explícita de tipos com verificação
  de perda por coerção.

Validação de integridade por reconciliação de contagem entre origem (CSV) e destino (Delta).

## 1. Configuração do ambiente e parâmetros de leitura

In [ ]:
# ---- AJUSTE ESTES VALORES ----
CSV_DIR = "/Volumes/certificacao_ae/adventure_works/adventureworks/AdventureWorks/data/"                 # mantenha o path que você já corrigiu
CATALOG = "workspace"
SCHEMA_BRONZE = "adventureworks_bronze"
SCHEMA_SILVER = "adventureworks_silver"

# Os arquivos do AdventureWorks são separados por TAB e NÃO têm linha de cabeçalho.
HAS_HEADER = False

READ_OPTS = {
    "header":    str(HAS_HEADER).lower(),   # False -> colunas recebem nomes genéricos _c0, _c1...
    "sep":       "\t",                       # separador TAB (a causa do erro)
    "quote":     '"',
    "escape":    '"',
    "multiLine": "true",
    "encoding":  "UTF-8",
}

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_BRONZE}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_SILVER}")
print("Config OK.")

## 2. Camada Bronze — ingestão fiel (todas as colunas como texto)

In [ ]:
from pyspark.sql import functions as F

# Camada Bronze: schema-on-read desabilitado (inferSchema omitido).
# Todas as colunas são lidas como string para garantir fidelidade ao dado de origem;
# a tipagem é responsabilidade da camada Silver.

arquivos = [f.name for f in dbutils.fs.ls(CSV_DIR) if f.name.lower().endswith(".csv")]

for nome in sorted(arquivos):
    caminho = f"{CSV_DIR}/{nome}"
    tabela  = f"{CATALOG}.{SCHEMA_BRONZE}.{nome[:-4].lower()}"
    try:
        df = spark.read.options(**READ_OPTS).csv(caminho)

        if len(df.columns) == 0:                     # arquivos vazios são ignorados
            print(f"[skip] vazio: {nome}")
            continue

        # Metadados de auditoria (linhagem e timestamp de ingestão)
        df = (df
              .withColumn("_arquivo_origem", F.lit(nome))
              .withColumn("_ingerido_em",    F.current_timestamp()))

        (df.write.format("delta").mode("overwrite")
           .option("overwriteSchema", "true")
           .saveAsTable(tabela))

        print(f"[ok] {tabela:55s} {df.count():>8} registros")
    except Exception as e:
        print(f"[erro] {nome}: {e}")

## 3. Validação — reconciliação de contagem (CSV × Delta)

In [ ]:
from pyspark.sql import functions as F

# Validação de integridade: reconciliação da contagem de registros entre origem e destino.
# n_csv    = contagem lógica (registros parseados)
# n_fisico = contagem de linhas físicas do arquivo, menos o cabeçalho (controle independente)
# n_delta  = registros persistidos na tabela Delta
# Divergência entre n_fisico e n_csv com n_delta consistente indica campos multiline.

resultados = []
for nome in sorted(arquivos):
    caminho = f"{CSV_DIR}/{nome}"
    tabela  = f"{CATALOG}.{SCHEMA_BRONZE}.{nome[:-4].lower()}"
    if not spark.catalog.tableExists(tabela):
        continue

    n_csv    = spark.read.options(**READ_OPTS).csv(caminho).count()
    n_fisico = spark.read.text(caminho).count() - (1 if HAS_HEADER else 0)
    n_delta  = spark.table(tabela).count()

    dif = n_csv - n_delta
    status = "ok" if dif == 0 else "divergencia"
    if dif == 0 and n_fisico != n_csv:
        status = "ok (multiline)"
    resultados.append((nome, n_csv, n_delta, dif, status))

df_rel = spark.createDataFrame(
    resultados,
    "arquivo string, linhas_csv long, linhas_delta long, diferenca long, status string")
display(df_rel.orderBy(F.when(F.col("status").startswith("ok"), 1).otherwise(0), "arquivo"))

## 4. Camada Silver — tipagem das tabelas centrais

In [ ]:
from pyspark.sql import functions as F

# Esquema POSICIONAL do AdventureWorks: a ordem tem que bater com a ordem das colunas no arquivo.
# (nome_final, tipo). Flags ficam como string por segurança até confirmarmos a codificação (0/1 vs true/false).
ESQUEMAS = {
    "salesorderheader": [
        ("SalesOrderID","int"),("RevisionNumber","int"),("OrderDate","timestamp"),
        ("DueDate","timestamp"),("ShipDate","timestamp"),("Status","int"),
        ("OnlineOrderFlag","string"),("SalesOrderNumber","string"),("PurchaseOrderNumber","string"),
        ("AccountNumber","string"),("CustomerID","int"),("SalesPersonID","int"),
        ("TerritoryID","int"),("BillToAddressID","int"),("ShipToAddressID","int"),
        ("ShipMethodID","int"),("CreditCardID","int"),("CreditCardApprovalCode","string"),
        ("CurrencyRateID","int"),("SubTotal","decimal(19,4)"),("TaxAmt","decimal(19,4)"),
        ("Freight","decimal(19,4)"),("TotalDue","decimal(19,4)"),("Comment","string"),
        ("rowguid","string"),("ModifiedDate","timestamp"),
    ],
    "salesorderdetail": [
        ("SalesOrderID","int"),("SalesOrderDetailID","int"),("CarrierTrackingNumber","string"),
        ("OrderQty","int"),("ProductID","int"),("SpecialOfferID","int"),
        ("UnitPrice","decimal(19,4)"),("UnitPriceDiscount","decimal(19,4)"),
        ("LineTotal","decimal(38,6)"),("rowguid","string"),("ModifiedDate","timestamp"),
    ],
    "customer": [
        ("CustomerID","int"),("PersonID","int"),("StoreID","int"),("TerritoryID","int"),
        ("AccountNumber","string"),("rowguid","string"),("ModifiedDate","timestamp"),
    ],
    "product": [
        ("ProductID","int"),("Name","string"),("ProductNumber","string"),("MakeFlag","string"),
        ("FinishedGoodsFlag","string"),("Color","string"),("SafetyStockLevel","int"),
        ("ReorderPoint","int"),("StandardCost","decimal(19,4)"),("ListPrice","decimal(19,4)"),
        ("Size","string"),("SizeUnitMeasureCode","string"),("WeightUnitMeasureCode","string"),
        ("Weight","decimal(8,2)"),("DaysToManufacture","int"),("ProductLine","string"),
        ("Class","string"),("Style","string"),("ProductSubcategoryID","int"),
        ("ProductModelID","int"),("SellStartDate","timestamp"),("SellEndDate","timestamp"),
        ("DiscontinuedDate","timestamp"),("rowguid","string"),("ModifiedDate","timestamp"),
    ],
}

def construir_silver(tabela, colunas):
    bronze = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.{tabela}")
    cols_dados = [c for c in bronze.columns if c.startswith("_c")]   # ignora colunas de auditoria

    # trava: a contagem de colunas do arquivo tem que bater com o esquema esperado
    if len(cols_dados) != len(colunas):
        print(f"[ATENCAO] {tabela}: arquivo tem {len(cols_dados)} colunas, "
              f"esquema esperava {len(colunas)}. Renomeacao abortada — me manda o print abaixo.")
        bronze.select(cols_dados).show(1, truncate=False)
        return

    # renomeia por posicao (_c0 -> 1a do esquema) e converte o tipo
    exprs = [F.expr(f"try_cast(_c{i} AS {tipo}) AS {nome}") for i,(nome,tipo) in enumerate(colunas)]
    silver = bronze.select(*exprs)

    # verificacao de perda por conversao (so avisa se algo nao converteu)
    for i,(nome,tipo) in enumerate(colunas):
        if tipo == "string": 
            continue
        falhas = bronze.where(F.col(f"_c{i}").isNotNull() &
                              F.expr(f"try_cast(_c{i} AS {tipo}) IS NULL")).count()
        if falhas:
            print(f"  [cast] {tabela}.{nome} ({tipo}): {falhas} valores nao converteram")

    (silver.write.format("delta").mode("overwrite").option("overwriteSchema","true")
           .saveAsTable(f"{CATALOG}.{SCHEMA_SILVER}.{tabela}"))
    print(f"[ok] silver.{tabela}: {silver.count()} registros, {len(colunas)} colunas")

for tabela, colunas in ESQUEMAS.items():
    construir_silver(tabela, colunas)

# Conferencia visual final: os nomes batem com o conteudo?
display(spark.table(f"{CATALOG}.{SCHEMA_SILVER}.salesorderheader").limit(5))

## 5. Camada Silver — tabelas complementares (geografia, motivo, cartão)

In [ ]:
# Tabelas adicionais necessarias para as perguntas de negocio (geografia, motivo, cartao)
ESQUEMAS_EXTRA = {
    "address": [
        ("AddressID","int"),("AddressLine1","string"),("AddressLine2","string"),
        ("City","string"),("StateProvinceID","int"),("PostalCode","string"),
        ("SpatialLocation","string"),("rowguid","string"),("ModifiedDate","timestamp"),
    ],
    "stateprovince": [
        ("StateProvinceID","int"),("StateProvinceCode","string"),("CountryRegionCode","string"),
        ("IsOnlyStateProvinceFlag","string"),("Name","string"),("TerritoryID","int"),
        ("rowguid","string"),("ModifiedDate","timestamp"),
    ],
    "countryregion": [
        ("CountryRegionCode","string"),("Name","string"),("ModifiedDate","timestamp"),
    ],
    "salesreason": [
        ("SalesReasonID","int"),("Name","string"),("ReasonType","string"),("ModifiedDate","timestamp"),
    ],
    "salesorderheadersalesreason": [
        ("SalesOrderID","int"),("SalesReasonID","int"),("ModifiedDate","timestamp"),
    ],
    "creditcard": [
        ("CreditCardID","int"),("CardType","string"),("CardNumber","string"),
        ("ExpMonth","int"),("ExpYear","int"),("ModifiedDate","timestamp"),
    ],
    "person": [
        ("BusinessEntityID","int"),("PersonType","string"),("NameStyle","string"),
        ("Title","string"),("FirstName","string"),("MiddleName","string"),
        ("LastName","string"),("Suffix","string"),("EmailPromotion","int"),
        ("AdditionalContactInfo","string"),("Demographics","string"),
        ("rowguid","string"),("ModifiedDate","timestamp"),
    ],
}

for tabela, colunas in ESQUEMAS_EXTRA.items():
    construir_silver(tabela, colunas)

### Garantir que os dados estão corretos

In [ ]:
from pyspark.sql import functions as F

B = f"{CATALOG}.{SCHEMA_BRONZE}"
S = f"{CATALOG}.{SCHEMA_SILVER}"

print("== 1. Nenhuma linha perdida na tipagem (bronze deve ser igual a silver) ==")
for t in ["salesorderheader","salesorderdetail","customer","product"]:
    b = spark.table(f"{B}.{t}").count()
    s = spark.table(f"{S}.{t}").count()
    print(f"  {t:20s} bronze={b:>8}  silver={s:>8}  {'OK' if b==s else '>>> DIVERGENCIA'}")

print("\n== 2. Chave primaria unica e sem nulo ==")
for t, pk in [("salesorderheader","SalesOrderID"),("salesorderdetail","SalesOrderDetailID"),
              ("customer","CustomerID"),("product","ProductID")]:
    df = spark.table(f"{S}.{t}")
    tot, dist = df.count(), df.select(pk).distinct().count()
    nulos = df.where(F.col(pk).isNull()).count()
    print(f"  {t:20s} {pk:18s} unica={'OK' if tot==dist else '>>> NAO'}  nulos={nulos}")

print("\n== 3. Integridade referencial (orfaos devem ser 0) ==")
soh = spark.table(f"{S}.salesorderheader")
sod = spark.table(f"{S}.salesorderdetail")
cus = spark.table(f"{S}.customer")
pro = spark.table(f"{S}.product")
print(f"  detalhes sem pedido correspondente : {sod.join(soh,'SalesOrderID','left_anti').count()}")
print(f"  pedidos sem cliente correspondente : {soh.join(cus,'CustomerID','left_anti').count()}")
print(f"  detalhes sem produto correspondente: {sod.join(pro,'ProductID','left_anti').count()}")

## 7. Validação de negócio — faturamento total × número informado pelo CEO

In [ ]:
# Validacao de negocio: o faturamento calculado bate com o numero informado pela lideranca (CEO)?
from pyspark.sql import functions as F

# >>> PREENCHA com o numero informado pelo CEO/desafio (apenas numero, sem R$ ou pontos de milhar).
#     Ex.: 109846381.40
VALOR_CEO = None

sod = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.salesorderdetail")
soh = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.salesorderheader")

# Tres formas de medir "vendas". Em AdventureWorks, LineTotal e SubTotal coincidem (receita liquida);
# TotalDue inclui imposto e frete. Uma delas deve casar com o numero do CEO.
faturamento_itens  = sod.agg(F.sum("LineTotal")).first()[0]   # receita por item (grao de item)
subtotal_pedidos   = soh.agg(F.sum("SubTotal")).first()[0]    # soma dos subtotais dos pedidos
total_com_impostos = soh.agg(F.sum("TotalDue")).first()[0]    # subtotal + imposto + frete

print("Faturamento (soma LineTotal, por item)   :", faturamento_itens)
print("Faturamento (soma SubTotal, por pedido)  :", subtotal_pedidos)
print("Total com imposto e frete (soma TotalDue):", total_com_impostos)

if VALOR_CEO is not None:
    print()
    for nome, valor in [("LineTotal", faturamento_itens),
                        ("SubTotal",  subtotal_pedidos),
                        ("TotalDue",  total_com_impostos)]:
        dif = float(valor) - float(VALOR_CEO)
        status = "BATE" if abs(dif) < 0.01 else f"difere em {dif:,.2f}"
        print(f"  vs CEO ({VALOR_CEO}) usando {nome}: {status}")
else:
    print("\n>>> Preencha VALOR_CEO acima com o numero informado para comparar automaticamente.")

### Iniciando o EDA




### EDA 1 — Estrutura e grão de cada tabela

In [ ]:
# Estrutura das tabelas da camada Silver: dimensoes (linhas x colunas) e tipos.
for t in ["salesorderheader","salesorderdetail","customer","product"]:
    df = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.{t}")
    print(f"\n=== {t} | {df.count()} linhas x {len(df.columns)} colunas ===")
    df.printSchema()
# Grao: salesorderheader = 1 pedido; salesorderdetail = 1 item de pedido;
#       customer = 1 cliente; product = 1 produto.

### EDA 2 — Completude (percentual de nulos por coluna)

In [ ]:
from pyspark.sql import functions as F

# Percentual de nulos por coluna: sinaliza colunas incompletas para tratamento.
for t in ["salesorderheader","salesorderdetail","customer","product"]:
    df = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.{t}")
    n = df.count()
    print(f"\n== {t} ==")
    nulos = df.select([ (F.count(F.when(F.col(c).isNull(), c))/n*100).alias(c) for c in df.columns ])
    display(nulos)

### EDA 3 — Estatísticas descritivas e testes de sanidade

In [ ]:
from pyspark.sql import functions as F

sod = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.salesorderdetail")
soh = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.salesorderheader")

# Estatisticas descritivas das medidas de venda
display(sod.select("OrderQty","UnitPrice","UnitPriceDiscount","LineTotal").summary())
display(soh.select("SubTotal","TaxAmt","Freight","TotalDue").summary())

# Sanidade: valores que nao deveriam existir (quantidade/valor <= 0)
print("Itens com quantidade <= 0 :", sod.where(F.col("OrderQty") <= 0).count())
print("Itens com preco <= 0      :", sod.where(F.col("UnitPrice") <= 0).count())
print("Pedidos com total < 0     :", soh.where(F.col("TotalDue") < 0).count())

### EDA 4 — Variáveis categóricas (frequência e cardinalidade)

In [ ]:
from pyspark.sql import functions as F

soh = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.salesorderheader")
pro = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.product")

# Frequencia de cada categoria. Cardinalidade alta demais indica dado sujo.
display(soh.groupBy("Status").count().orderBy("Status"))
display(soh.groupBy("OnlineOrderFlag").count())
display(soh.groupBy("TerritoryID").count().orderBy(F.desc("count")))
display(pro.groupBy("ProductLine").count().orderBy(F.desc("count")))
display(pro.groupBy("Color").count().orderBy(F.desc("count")))

### EDA 5 — Análise temporal (faixa de datas e receita por mês)

In [ ]:
from pyspark.sql import functions as F

soh = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.salesorderheader")

# Faixa de datas: os limites fazem sentido?
soh.select(F.min("OrderDate").alias("primeira"), F.max("OrderDate").alias("ultima")).show()

# Volume e receita por mes (clique no icone de grafico no resultado para ver a curva)
por_mes = (soh
    .groupBy(F.date_format("OrderDate","yyyy-MM").alias("ano_mes"))
    .agg(F.count("*").alias("qtd_pedidos"), F.round(F.sum("TotalDue"),2).alias("receita"))
    .orderBy("ano_mes"))
display(por_mes)

### EDA 6 — Rankings de produtos e clientes

In [ ]:
from pyspark.sql import functions as F

soh = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.salesorderheader")
sod = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.salesorderdetail")
pro = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.product")

# Top 10 produtos por receita (junta item de venda com produto)
top_prod = (sod.join(pro, "ProductID")
    .groupBy("ProductID","Name")
    .agg(F.round(F.sum("LineTotal"),2).alias("receita"), F.sum("OrderQty").alias("unidades"))
    .orderBy(F.desc("receita")).limit(10))
display(top_prod)

# Top 10 clientes por receita
top_cli = (soh.groupBy("CustomerID")
    .agg(F.round(F.sum("TotalDue"),2).alias("receita"), F.count("*").alias("qtd_pedidos"))
    .orderBy(F.desc("receita")).limit(10))
display(top_cli)